In [0]:
import os
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType

# 1. SECURITY: Hiding credentials in Environment Variables 
# (This helps bypass strict security checks on Shared clusters)
os.environ["K_USER"] = "7JCBLXM7QK3FIOKP"
os.environ["K_PASS"] = "cfltJHNzFz9th/5yClNbOWk8s3gZFFytEsf8Puxw3ZTlFbNmaR0yNUKYyl0cGBrg"

# 2. SCHEMA: Defining the structure of our JSON "Clicks"
schema = StructType() \
    .add("user_id", IntegerType()) \
    .add("event_type", StringType()) \
    .add("item_id", StringType()) \
    .add("price", DoubleType()) \
    .add("traffic_source", StringType()) \
    .add("ip_address", StringType()) \
    .add("event_timestamp", StringType())

# 3. CONFIGURATION: Connecting to Confluent Cloud
# Note: Using 'kafkashaded' and singular 'mechanism' for Shared clusters
kafka_options = {
    "kafka.bootstrap.servers": "pkc-619z3.us-east1.gcp.confluent.cloud:9092",
    "subscribe": "user_events",
    "startingOffsets": "latest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN", 
    "kafka.sasl.jaas.config": f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{os.environ['K_USER']}' password='{os.environ['K_PASS']}';"
}

# 4. THE STREAM: Reading live data from Kafka
raw_stream_df = spark.readStream \
    .format("kafka") \
    .options(**kafka_options) \
    .load()

# 5. TRANSFORMATION: Converting Binary bytes into a readable Table
# Kafka stores data as binary. We cast it to String, then parse the JSON.
clean_stream_df = raw_stream_df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")

# 6. OUTPUT: Displaying the live-updating table
query = clean_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/my_volume/checkpoints/kafka_v1") \
    .trigger(availableNow=True) \
    .start("/Volumes/workspace/default/my_volume/data/kafka_output")

query.awaitTermination()


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS my_volume;

In [0]:
SHOW CATALOGS;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.default;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.my_volume;

In [0]:
# 1. Read the Delta table where your Consumer just saved the data
df_saved_messages = spark.read.format("delta").load("/Volumes/workspace/default/my_volume/data/kafka_output")

# 2. Show the data on the screen!
display(df_saved_messages)


In [0]:
# Notice we use readStream here instead of read
live_bronze_df = spark.readStream \
    .format("delta") \
    .load("/Volumes/workspace/default/my_volume/data/kafka_output")

# Add a unique checkpoint path just for this display view
display(live_bronze_df, checkpointLocation="/Volumes/workspace/default/my_volume/checkpoints/display_bronze_v1")


In [0]:
from pyspark.sql.functions import col, window, count

# 1. Read the live Bronze table we just built
bronze_stream = spark.readStream \
    .format("delta") \
    .load("/Volumes/workspace/default/my_volume/data/kafka_output")

# 2. Cast the timestamp string into a real Timestamp data type
timestamped_df = bronze_stream.withColumn("event_timestamp", col("event_timestamp").cast("timestamp"))

# 3. FRAUD LOGIC: Group by 10-second windows and IP Address
fraud_alerts = timestamped_df \
    .withWatermark("event_timestamp", "1 minute") \
    .groupBy(
        window(col("event_timestamp"), "10 seconds"),
        col("ip_address")
    ) \
    .agg(count("*").alias("clicks_per_10_sec")) \
    .filter(col("clicks_per_10_sec") >= 4) # The Rule: 4 or more clicks in 10s is a BOT!

# 4. Display the live "Fraud Dashboard" (Notice the new checkpoint path!)
display(fraud_alerts, checkpointLocation="/Volumes/workspace/default/my_volume/checkpoints/fraud_alerts_v1")
